In [80]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

In [81]:
def load_fred(path, col_name):
    df = pd.read_csv(path, parse_dates=['observation_date'])
    df.columns = ['date', col_name]
    df[col_name] = pd.to_numeric(df[col_name], errors='coerce')
    df = df.set_index('date')
    df = df.dropna()
    return df

In [82]:
houst1f   = load_fred('HOUST1F.csv',        'housing_starts')
msacsr    = load_fred('MSACSR.csv',          'months_of_supply')
actlis    = load_fred('ACTLISCOUUS.csv',     'active_listings')
csushpisa = load_fred('CSUSHPISA.csv',       'home_price_index')
mspus     = load_fred('MSPUS.csv',           'median_home_price')
mortgage  = load_fred('MORTGAGE30US.csv',    'mortgage_rate')

In [83]:
print(houst1f.head())
print(msacsr.head())
print(actlis.head())
print(csushpisa.head())
print(mspus.head())
print(mortgage.head())

            housing_starts
date                      
2000-01-01            1268
2000-02-01            1255
2000-03-01            1313
2000-04-01            1275
2000-05-01            1230
            months_of_supply
date                        
2000-01-01               4.3
2000-02-01               4.3
2000-03-01               4.3
2000-04-01               4.4
2000-05-01               4.4
            active_listings
date                       
2016-07-01          1463025
2016-08-01          1460071
2016-09-01          1443103
2016-10-01          1407722
2016-11-01          1339724
            home_price_index
date                        
2000-01-01           100.551
2000-02-01           101.339
2000-03-01           102.126
2000-04-01           102.922
2000-05-01           103.677
            median_home_price
date                         
2000-01-01             165300
2000-04-01             163200
2000-07-01             168800
2000-10-01             172900
2001-01-01             169800

In [84]:
# Turn weekly to monthly value
mortgage = mortgage.resample('MS').mean()
print(mortgage)
#Turn quarter to monthly value (forward fill)
mspus = mspus.resample('MS').ffill()
print(mspus)

            mortgage_rate
date                     
2000-01-01         8.2100
2000-02-01         8.3250
2000-03-01         8.2400
2000-04-01         8.1525
2000-05-01         8.5150
...                   ...
2025-12-01         6.1900
2026-01-01         6.1025
2026-02-01         6.0475
2026-03-01         6.1775
2026-04-01         6.3320

[316 rows x 1 columns]
            median_home_price
date                         
2000-01-01             165300
2000-02-01             165300
2000-03-01             165300
2000-04-01             163200
2000-05-01             163200
...                       ...
2025-06-01             416100
2025-07-01             410100
2025-08-01             410100
2025-09-01             410100
2025-10-01             405300

[310 rows x 1 columns]


In [85]:
# Merge ALL together...
merge = houst1f.merge(msacsr, how = 'outer', on = 'date')
merge = merge.merge(actlis, how = 'outer', on = 'date')
merge = merge.merge(csushpisa, how = 'outer', on = 'date')
merge = merge.merge(mspus, how = 'outer', on = 'date')
merge = merge.merge(mortgage, how = 'outer', on = 'date')
print(merge)

            housing_starts  months_of_supply  active_listings  \
date                                                            
2000-01-01          1268.0               4.3              NaN   
2000-02-01          1255.0               4.3              NaN   
2000-03-01          1313.0               4.3              NaN   
2000-04-01          1275.0               4.4              NaN   
2000-05-01          1230.0               4.4              NaN   
...                    ...               ...              ...   
2025-12-01           950.0               8.0         976833.0   
2026-01-01           898.0               9.7         912696.0   
2026-02-01           941.0               NaN         915717.0   
2026-03-01          1032.0               NaN         947866.0   
2026-04-01             NaN               NaN        1002935.0   

            home_price_index  median_home_price  mortgage_rate  
date                                                            
2000-01-01           100

In [ ]:
merge = merge[merge.index <=  '2025-10-01' ] #Keep from 2000 to 10/2025 only
print(merge)

            housing_starts  months_of_supply  active_listings  \
date                                                            
2000-01-01          1268.0               4.3              NaN   
2000-02-01          1255.0               4.3              NaN   
2000-03-01          1313.0               4.3              NaN   
2000-04-01          1275.0               4.4              NaN   
2000-05-01          1230.0               4.4              NaN   
...                    ...               ...              ...   
2025-06-01           925.0               9.1        1082520.0   
2025-07-01           951.0               9.3        1102787.0   
2025-08-01           869.0               8.4        1098681.0   
2025-09-01           836.0               8.1        1100407.0   
2025-10-01           894.0               9.0        1099856.0   

            home_price_index  median_home_price  mortgage_rate  
date                                                            
2000-01-01           100

In [ ]:
merge.to_csv("housing_dashboard_data.csv") #Export to csv file

In [86]:
# CLAIM 1: Homebuilding collapsed after 2008 and never recovered
print("\n" + "="*60)
print("CLAIM 1: Housing starts collapsed and never recovered")
print("="*60)
 
pre_crisis_peak  = houst1f.loc['2000':'2006','housing_starts'].max()
crisis_trough    = houst1f.loc['2008':'2012','housing_starts'].min()
recent_avg       = houst1f.loc['2020':'2024','housing_starts'].mean()
pre_crisis_avg   = houst1f.loc['2000':'2006','housing_starts'].mean()
pct_drop         = (crisis_trough - pre_crisis_peak) / pre_crisis_peak * 100
pct_recovery     = (recent_avg - pre_crisis_avg) / pre_crisis_avg * 100
 
print(f"  Pre-crisis peak (2000-2006):       {pre_crisis_peak:,.0f}k units/yr")
print(f"  Crisis trough (2008-2012):         {crisis_trough:,.0f}k units/yr")
print(f"  Drop from peak to trough:          {pct_drop:.1f}%")
print(f"  Pre-crisis avg (2000-2006):        {pre_crisis_avg:,.0f}k units/yr")
print(f"  Recent avg (2020-2024):            {recent_avg:,.0f}k units/yr")
print(f"  Recovery vs pre-crisis avg:        {pct_recovery:.1f}%")
print(f"  VERDICT: Still {abs(pct_recovery):.0f}% below pre-crisis average" if pct_recovery < 0
      else f"  VERDICT: Fully recovered (+{pct_recovery:.0f}%)")


CLAIM 1: Housing starts collapsed and never recovered
  Pre-crisis peak (2000-2006):       1,823k units/yr
  Crisis trough (2008-2012):         353k units/yr
  Drop from peak to trough:          -80.6%
  Pre-crisis avg (2000-2006):        1,453k units/yr
  Recent avg (2020-2024):            1,019k units/yr
  Recovery vs pre-crisis avg:        -29.8%
  VERDICT: Still 30% below pre-crisis average


In [87]:
# CLAIM 2: Supply shortage — inventory at historic lows
